# Explicações sobre o que e porque estamos fazendo pra clusterizar:

Disclaimer importante: os dados são composicionais isso é um problema, mas estamos trabalhandoc omo se fossem independentes

LPA, Latent Profile Analysis, é uma forma de clusterização estatística/probabilística. A ideia central é:

> existem grupos “latentes” não observados diretamente na população, e os dados observados de cada município são gerados por um desses perfis.

O LPA tenta inferir esse “tipo” como uma variável categórica escondida.

$$z_i ∈ {1, 2, 3, ..., K}$$

Onde K é o numero de perfis inferidos, então ele calcula a probabilidade de petencimento da amostra a cada perfil.

$$
P(z_i = 1 | \text{ dados do município})\\
P(z_i = 2 | \text{ dados do município})\\
P(z_i = 3 | \text{ dados do município})\\
\cdots\\
P(z_i = K | \text{ dados do município})\\
$$

Matematicamente o que modelo tenta fazer é:

Dado um vetor de estado $x_i = [x_i1, x_i2, x_i3, ..., x_ip]$, onde i é a amostra e p a quantidade de variáveis

O LPA assume que a distribuição dos dados é uma mistura de $K$ perfis:

$$p(x_i)_h = \pi_h N(x_i \mid \mu_h, \Sigma_h)$$

* $\pi_h$ = proporção esperada da população no perfil $h$
* $\mu_h$ = media de $h$
* $\sum_K$ = covariância de $k$
* $N$ é a normal multivariada

E então para cada município:

$$P(z_i=k \mid x_i) = \frac{p(x_i)_k}{\sum^{K}_{j=1}p(x_i)_j}$$

e usando as probabilidade encontradas, atualizamos $\pi_k$, $\mu_k$ e $\Sigma_K$

e repete até convergir suficientemente

Referências

[Sinha P, Calfee CS, Delucchi KL. Practitioner's Guide to Latent Class Analysis: Methodological Considerations and Common Pitfalls. Crit Care Med. 2021 Jan 1;49(1):e63-e79. doi: 10.1097/CCM.0000000000004710. PMID: 33165028; PMCID: PMC7746621.](https://pmc.ncbi.nlm.nih.gov/articles/PMC7746621/?)



## Métricas extraidas de `src/clustering_met.Ipynb`

### dist_idade

`dist_idade_total`
menor BIC: 264962.42618618417 | k = 3.0
menor AIC: 254099.9844062482 | k = 20.0

`dist_idade_gen`
menor BIC: -576947.1746652153 | k = 2.0
menor AIC: -593028.3990586596 | k = 6.0


### dist_renda

`dist_renda_total`
menor BIC: 117863.41759762166 | k = 11.0
menor AIC: 112709.61356003738 | k = 18.0

`dist_renda_gen`
menor BIC: -194810.23327200292 | k = 6.0
menor AIC: -218983.0361354256 | k = 20.0

### dist_instucao

`dist_instucao_total`
menor BIC: 45407.31177094128 | k = 6.0
menor AIC: 44348.888524174996 | k = 23.0

`dist_instucao_gen`
menor BIC: -77138.93729970285 | k = 7.0
menor AIC: -80054.38277507952 | k = 22.0

Eu gostaria de manter os cluters com o mesmo número de clusters, parece mais estável e informativo usar as distribuições por gênero com k = 6

In [22]:
import numpy as np
import pandas as pd
import os
from pathlib import Path
from typing import Literal

from sklearn.mixture import GaussianMixture
from sklearn.base import BaseEstimator

from sql_utils import build_postgres_engine

covariance_type:Literal="full"
k = 6

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent

engine = build_postgres_engine(
        "localhost",
        int(os.environ.get("POSTGRES_PORT", 5432)),
        os.environ["POSTGRES_DB"],
        os.environ["POSTGRES_USER"],
        os.environ["POSTGRES_PASSWORD"],
    )

norm_schema = "norm_por_1k"


In [23]:
def prepare_lpa_matrix(
    df: pd.DataFrame,
    data_cols: list[str],
    divide_by: float | None = 1000,
    fillna_value: float = 0.0,
) -> pd.DataFrame:
    """
    Prepara a matriz usada no LPA/GMM.

    Parâmetros
    ----------
    df:
        DataFrame original.

    data_cols:
        Colunas que representam o vetor de dados de cada observação.
        Exemplo: faixas de renda, faixas de escolaridade ou faixas de idade.

    divide_by:
        Valor usado para converter os dados.
        Se os dados estão em pessoas por mil habitantes, use 1000.
        Se já estão em proporção, use None.

    fillna_value:
        Valor usado para preencher NaN.

    Retorna
    -------
    X:
        DataFrame apenas com as colunas do modelo, tratado e numérico.
    """
    missing_cols = [col for col in data_cols if col not in df.columns]

    if missing_cols:
        raise KeyError(f"As seguintes colunas não existem no DataFrame: {missing_cols}")

    X = df[data_cols].copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(fillna_value)
    X = X.astype(float)

    if divide_by is not None:
        X = X / divide_by

    return X


def fit_lpa_model(
    df: pd.DataFrame,
    data_cols: list[str],
    n_profiles: int,
    divide_by: float | None = None,
    covariance_type: str = "diag",
    random_state: int = 42,
    n_init: int = 20,
    max_iter: int = 1000,
    reg_covar: float = 1e-6,
) -> GaussianMixture:
    """
    Ajusta um modelo de Latent Profile Analysis usando GaussianMixture.

    Parâmetros
    ----------
    df:
        DataFrame original.

    data_cols:
        Colunas usadas no vetor de dados.

    n_profiles:
        Número de perfis latentes.

    divide_by:
        Use 1000 se os dados estão em pessoas por mil habitantes.
        Use None se os dados já estão em proporção.

    covariance_type:
        Tipo de covariância do GMM.
        Recomendado para começar: "diag".

    Retorna
    -------
    model:
        Modelo GaussianMixture ajustado.
    """
    X = prepare_lpa_matrix(
        df=df,
        data_cols=data_cols,
        divide_by=divide_by,
    )

    model = GaussianMixture(
        n_components=n_profiles,
        covariance_type=covariance_type,
        random_state=random_state,
        n_init=n_init,
        max_iter=max_iter,
        reg_covar=reg_covar,
    )

    model.fit(X)

    return model

In [30]:
import pandas as pd
import numpy as np


def filter_rows_by_coordinate_sum(
    df: pd.DataFrame,
    reference_column: str,
    coordinate_columns: list[str],
    expected_sum: float = 1000,
    sum_tolerance: float = 1e-6,
) -> tuple[pd.DataFrame, pd.Series]:
    """
    Separa o DataFrame em:
    - valid_df: linhas em que as colunas de coordenadas somam expected_sum;
    - excluded_df: linhas excluídas, com a soma calculada.

    Use isso antes de passar o DataFrame para o test_gmm.
    """

    required_columns = [reference_column, *coordinate_columns]

    missing_columns = [
        column for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise KeyError(
            f"As seguintes colunas não existem no DataFrame: {missing_columns}"
        )

    coordinate_sum = df[coordinate_columns].sum(axis=1)

    valid_mask = np.isclose(
        coordinate_sum,
        expected_sum,
        atol=sum_tolerance,
        rtol=0,
    )

    valid_df = df.loc[valid_mask].copy()

    excluded_df = df.loc[~valid_mask, [reference_column, *coordinate_columns]].copy()
    excluded_df["coordinate_sum"] = coordinate_sum.loc[~valid_mask]

    return valid_df, excluded_df

In [46]:
def add_lpa_profile_columns(
    df: pd.DataFrame,
    model: BaseEstimator,
    id_col: str,
    data_cols: list[str],
    prefix: str,
    divide_by: float | None = 1000,
    fillna_value: float = 0.0,
    copy: bool = True,
) -> pd.DataFrame:
    """
    Adiciona ao DataFrame as colunas de resultado do LPA/GMM.

    Colunas criadas
    ---------------
    {prefix}_perfil:
        Perfil mais provável.

    {prefix}_prob_max:
        Probabilidade do perfil mais provável.

    {prefix}_entropia_bruta:
        Entropia de Shannon bruta da distribuição de probabilidades.
        Varia de 0 até log(k), onde k é o número de perfis.
        Quanto menor, mais certa é a classificação.

    {prefix}_incerteza_normalizada:
        Entropia bruta dividida por log(k).
        Varia de 0 até 1.
        Quanto menor, mais certa é a classificação.

    {prefix}_entropia_classificacao:
        1 - incerteza_normalizada.
        Varia de 0 até 1.
        Quanto maior, mais certa é a classificação.
        Essa é mais parecida com a "entropy" usualmente reportada em LPA/LCA.

    {prefix}_prob_perfil_0, {prefix}_prob_perfil_1, ...
        Probabilidade de pertencimento a cada perfil.

    Parâmetros
    ----------
    df:
        DataFrame original.

    model:
        Modelo GaussianMixture já ajustado.

    id_col:
        Coluna de identificação da observação.
        Exemplo: código do município.

    data_cols:
        Colunas usadas no vetor de dados.

    prefix:
        Prefixo das colunas criadas.
        Exemplo: "renda", "escolaridade", "idade".

    divide_by:
        Use o mesmo valor usado no ajuste do modelo.

    fillna_value:
        Valor usado para preencher NaN antes de aplicar o modelo.

    copy:
        Se True, retorna uma cópia do DataFrame.
        Se False, modifica o DataFrame original.

    Retorna
    -------
    result_df:
        DataFrame com as novas colunas.
    """
    if id_col not in df.columns:
        raise KeyError(f"A coluna de ID '{id_col}' não existe no DataFrame.")

    result_df = df.copy() if copy else df

    X = prepare_lpa_matrix(
        df=result_df,
        data_cols=data_cols,
        divide_by=divide_by,
        fillna_value=fillna_value,
    )

    labels = model.predict(X)
    probs = model.predict_proba(X)

    n_profiles = probs.shape[1]

    max_probs = probs.max(axis=1)

    entropy_raw = -np.sum(probs * np.log(probs + 1e-12), axis=1)

    if n_profiles <= 1:
        normalized_uncertainty = np.zeros_like(entropy_raw)
        classification_entropy = np.ones_like(entropy_raw)
    else:
        normalized_uncertainty = entropy_raw / np.log(n_profiles)
        classification_entropy = 1 - normalized_uncertainty

    result_df[f"{prefix}_perfil"] = labels
    result_df[f"{prefix}_prob_max"] = max_probs

    result_df[f"{prefix}_entropia_bruta"] = entropy_raw
    result_df[f"{prefix}_incerteza_normalizada"] = normalized_uncertainty
    result_df[f"{prefix}_entropia_classificacao"] = classification_entropy

    for k in range(n_profiles):
        result_df[f"{prefix}_prob_perfil_{k}"] = probs[:, k]

    return result_df

In [25]:
import pandas as pd

def pivot_multiplas_colunas_populacao(
    df: pd.DataFrame,
    categoria_col: str,
    populacao_cols: list[str],
    rename_categories: dict | None = None,
    aggfunc: str = "sum",
    fill_value=0,
) -> pd.DataFrame:
    """
    Pivot várias colunas de população ao mesmo tempo.

    Preserva todas as colunas que NÃO sejam:
    - a coluna de categoria usada no pivot;
    - as colunas de população listadas em populacao_cols.

    Parâmetros
    ----------
    df:
        DataFrame original.

    categoria_col:
        Coluna que separa as linhas.
        Exemplo: "sexo", "genero", "tipo_populacao".

    populacao_cols:
        Lista de colunas populacionais que devem ser pivotadas.
        Exemplo: ["pop_0_4", "pop_5_9", "pop_10_14"].

    rename_categories:
        Dicionário opcional para renomear valores da categoria.
        Exemplo:
        {
            "Homens": "homens",
            "Mulheres": "mulheres"
        }

    aggfunc:
        Função de agregação caso exista mais de uma linha para o mesmo grupo.
        Normalmente "sum".

    fill_value:
        Valor usado quando uma combinação não existir.
    """

    if rename_categories is None:
        rename_categories = {}

    # Colunas de identificação que serão preservadas
    id_cols = [
        col for col in df.columns
        if col not in [categoria_col, *populacao_cols]
    ]

    # Faz o pivot com múltiplas colunas de valores
    pivot = df.pivot_table(
        index=id_cols,
        columns=categoria_col,
        values=populacao_cols,
        aggfunc=aggfunc,
        fill_value=fill_value,
    )

    # Achata o MultiIndex das colunas
    # Exemplo: ("pop_0_4", "Homens") -> "pop_0_4_homens"
    pivot.columns = [
        f"{pop_col}_{rename_categories.get(categoria, str(categoria))}"
        for pop_col, categoria in pivot.columns
    ]

    pivot = pivot.reset_index()

    return pivot

In [32]:
dist_idade = pd.read_sql_table("dist_idade", con=engine, schema=norm_schema)
dist_idade = dist_idade.fillna(0)



dist_idade, excluidos = filter_rows_by_coordinate_sum(
    df = dist_idade, 
    reference_column="cod_territorio",
    coordinate_columns=['0_4', '10_14',
       '15_19', '20_24', '25_29', '30_34', '35_39', '40_44', '45_49', '50_54',
       '55_59', '5_9', '60_64', '65_69', '70_74', '75_79', '80_84', '85_89',
       '90_94', '95_99', '100_mais'],
    sum_tolerance= 2
)
excluidos

,cod_territorio,0_4,10_14,15_19,20_24,25_29,30_34,35_39,40_44,45_49,...,60_64,65_69,70_74,75_79,80_84,85_89,90_94,95_99,100_mais,coordinate_sum


In [33]:
dist_idade = dist_idade[dist_idade["cod_territorio"].astype(int) > 100].copy()
dist_idade = dist_idade[dist_idade["populacao"] != "total"].copy()

dist_idade_gen = pivot_multiplas_colunas_populacao(
    df=dist_idade,
    categoria_col='populacao',
    populacao_cols=[
    '0_4', '10_14',
    '15_19', '20_24', 
    '25_29', '30_34', 
    '35_39', '40_44', 
    '45_49', '50_54',
    '55_59', '5_9', 
    '60_64', '65_69', 
    '70_74', '75_79', 
    '80_84', '85_89',
    '90_94', '95_99', 
    '100_mais'
    ],
    rename_categories={
        "Homens": "homens",
        "Mulheres": "mulheres",
    },
)

In [34]:
dist_idade_gen.columns

Index(['id', 'cod_territorio', 'territorio', '0_4_homens', '0_4_mulheres',
       '100_mais_homens', '100_mais_mulheres', '10_14_homens',
       '10_14_mulheres', '15_19_homens', '15_19_mulheres', '20_24_homens',
       '20_24_mulheres', '25_29_homens', '25_29_mulheres', '30_34_homens',
       '30_34_mulheres', '35_39_homens', '35_39_mulheres', '40_44_homens',
       '40_44_mulheres', '45_49_homens', '45_49_mulheres', '50_54_homens',
       '50_54_mulheres', '55_59_homens', '55_59_mulheres', '5_9_homens',
       '5_9_mulheres', '60_64_homens', '60_64_mulheres', '65_69_homens',
       '65_69_mulheres', '70_74_homens', '70_74_mulheres', '75_79_homens',
       '75_79_mulheres', '80_84_homens', '80_84_mulheres', '85_89_homens',
       '85_89_mulheres', '90_94_homens', '90_94_mulheres', '95_99_homens',
       '95_99_mulheres'],
      dtype='str')

In [47]:
dim_cols = [
    '0_4_homens', '0_4_mulheres', '95_99_mulheres',
    '100_mais_homens', '100_mais_mulheres', '10_14_homens',
    '10_14_mulheres', '15_19_homens', '15_19_mulheres', '20_24_homens',
    '20_24_mulheres', '25_29_homens', '25_29_mulheres', '30_34_homens',
    '30_34_mulheres', '35_39_homens', '35_39_mulheres', '40_44_homens',
    '40_44_mulheres', '45_49_homens', '45_49_mulheres', '50_54_homens',
    '50_54_mulheres', '55_59_homens', '55_59_mulheres', '5_9_homens',
    '5_9_mulheres', '60_64_homens', '60_64_mulheres', '65_69_homens',
    '65_69_mulheres', '70_74_homens', '70_74_mulheres', '75_79_homens',
    '75_79_mulheres', '80_84_homens', '80_84_mulheres', '85_89_homens',
    '85_89_mulheres', '90_94_homens', '90_94_mulheres', '95_99_homens'
]

id_col = "cod_territorio"

c_dist_idade = add_lpa_profile_columns(
    df=dist_idade_gen,
    id_col=id_col,
    data_cols=dim_cols,
    prefix="idade",
    divide_by=1000,
    model=fit_lpa_model(
        df=dist_idade_gen,
        data_cols=dim_cols,
        n_profiles=6,
        divide_by=1000,
        covariance_type=covariance_type,
    ),
)

In [51]:
print(f"idade_entropia_classificacao: {c_dist_idade["idade_entropia_classificacao"].mean()}")
print(f"idade_incerteza_normalizada: {c_dist_idade["idade_incerteza_normalizada"].mean()}")
print(f"idade_entropia_bruta: {c_dist_idade["idade_entropia_bruta"].mean()}")
print(f"idade_prob_max: {c_dist_idade["idade_prob_max"].mean()}")

idade_entropia_classificacao: 0.9064218242782169
idade_incerteza_normalizada: 0.09357817572178304
idade_entropia_bruta: 0.16766958246259164
idade_prob_max: 0.9339006455162067


In [49]:
c_dist_idade

,id,cod_territorio,territorio,0_4_homens,0_4_mulheres,100_mais_homens,100_mais_mulheres,10_14_homens,10_14_mulheres,15_19_homens,...,idade_prob_max,idade_entropia_bruta,idade_incerteza_normalizada,idade_entropia_classificacao,idade_prob_perfil_0,idade_prob_perfil_1,idade_prob_perfil_2,idade_prob_perfil_3,idade_prob_perfil_4,idade_prob_perfil_5
0,11000151,1100015,Alta Floresta D'Oeste (RO),73.404648,0.000000,0.092217,0.000000,73.681298,0.000000,79.767613,...,0.923410,0.270455,0.150944,0.849056,0.000000,0.923410,0.000000,0.076580,0.000000,9.450628e-06
1,11000152,1100015,Alta Floresta D'Oeste (RO),0.000000,72.112676,0.000000,0.000000,0.000000,69.671362,0.000000,...,0.891101,0.419681,0.234229,0.765771,0.891101,0.000000,0.055188,0.000000,0.053711,0.000000e+00
2,11000231,1100023,Ariquemes (RO),72.705676,0.000000,0.041677,0.000000,73.768442,0.000000,80.478453,...,0.864369,0.442608,0.247024,0.752976,0.000000,0.864369,0.000000,0.121347,0.000000,1.428426e-02
3,11000232,1100023,Ariquemes (RO),0.000000,70.774900,0.000000,0.061419,0.000000,69.955983,0.000000,...,0.622642,0.718794,0.401166,0.598834,0.364514,0.000000,0.622642,0.000000,0.012843,0.000000e+00
4,11000491,1100049,Cacoal (RO),67.815335,0.000000,0.164543,0.000000,70.871139,0.000000,74.514597,...,0.908661,0.306880,0.171273,0.828727,0.000000,0.908661,0.000000,0.091166,0.000000,1.727587e-04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3415,52216012,5221601,Uruaçu (GO),0.000000,62.450922,0.000000,0.461915,0.000000,65.176221,0.000000,...,0.786644,0.519145,0.289741,0.710259,0.213268,0.000000,0.786644,0.000000,0.000088,0.000000e+00
3416,52218581,5221858,Valparaíso de Goiás (GO),79.944441,0.000000,0.010443,0.000000,79.067193,0.000000,79.725129,...,0.949015,0.201405,0.112406,0.887594,0.000000,0.050985,0.000000,0.949015,0.000000,2.791923e-11
3417,52218582,5221858,Valparaíso de Goiás (GO),0.000000,69.975850,0.000000,0.038795,0.000000,71.304567,0.000000,...,0.535380,0.691742,0.386068,0.613932,0.000119,0.000000,0.535380,0.000000,0.464501,0.000000e+00
3418,53001081,5300108,Brasília (DF),63.007806,0.000000,0.057343,0.000000,69.262712,0.000000,78.528522,...,0.921225,0.290628,0.162203,0.837797,0.000000,0.921225,0.000000,0.075098,0.000000,3.677744e-03


In [54]:
dist_renda = pd.read_sql_table("dist_renda_T", con=engine, schema=norm_schema)
dist_renda = dist_renda.fillna(0)

dist_renda = dist_renda.drop(columns=["Total"])

dist_renda, excluidos = filter_rows_by_coordinate_sum(
    df = dist_renda, 
    reference_column="Cod",
    coordinate_columns=['Até 1/4 de salário mínimo', 'Mais de 1 a 2 salários mínimos',
       'Mais de 1/2 a 1 salário mínimo', 'Mais de 1/4 a 1/2 salário mínimo',
       'Mais de 10 a 15 salários mínimos', 'Mais de 15 a 20 salários mínimos',
       'Mais de 2 a 3 salários mínimos', 'Mais de 20 salários mínimos',
       'Mais de 3 a 5 salários mínimos', 'Mais de 5 a 10 salários mínimos',
       'Sem rendimento'],
    sum_tolerance= 2
)
excluidos

,Cod,Até 1/4 de salário mínimo,Mais de 1 a 2 salários mínimos,Mais de 1/2 a 1 salário mínimo,Mais de 1/4 a 1/2 salário mínimo,Mais de 10 a 15 salários mínimos,Mais de 15 a 20 salários mínimos,Mais de 2 a 3 salários mínimos,Mais de 20 salários mínimos,Mais de 3 a 5 salários mínimos,Mais de 5 a 10 salários mínimos,Sem rendimento,coordinate_sum


In [55]:
dist_renda_gen = dist_renda[dist_renda["grupo"] != "Total"].copy()

In [56]:
dist_renda_gen = pivot_multiplas_colunas_populacao(
    df= dist_renda_gen,
    categoria_col="grupo",
    populacao_cols=['Até 1/4 de salário mínimo', 'Mais de 1 a 2 salários mínimos',
       'Mais de 1/2 a 1 salário mínimo', 'Mais de 1/4 a 1/2 salário mínimo',
       'Mais de 10 a 15 salários mínimos', 'Mais de 15 a 20 salários mínimos',
       'Mais de 2 a 3 salários mínimos', 'Mais de 20 salários mínimos',
       'Mais de 3 a 5 salários mínimos', 'Mais de 5 a 10 salários mínimos',
       'Sem rendimento'],
    rename_categories={
        "Homens": "homens",
        "Mulheres": "mulheres",
    },
)

In [58]:
dim_cols = [
    'Até 1/4 de salário mínimo_homens',
    'Até 1/4 de salário mínimo_mulheres',
    'Mais de 1 a 2 salários mínimos_homens',
    'Mais de 1 a 2 salários mínimos_mulheres',
    'Mais de 1/2 a 1 salário mínimo_homens',
    'Mais de 1/2 a 1 salário mínimo_mulheres',
    'Mais de 1/4 a 1/2 salário mínimo_homens',
    'Mais de 1/4 a 1/2 salário mínimo_mulheres',
    'Mais de 10 a 15 salários mínimos_homens',
    'Mais de 10 a 15 salários mínimos_mulheres',
    'Mais de 15 a 20 salários mínimos_homens',
    'Mais de 15 a 20 salários mínimos_mulheres',
    'Mais de 2 a 3 salários mínimos_homens',
    'Mais de 2 a 3 salários mínimos_mulheres',
    'Mais de 20 salários mínimos_homens',
    'Mais de 20 salários mínimos_mulheres',
    'Mais de 3 a 5 salários mínimos_homens',
    'Mais de 3 a 5 salários mínimos_mulheres',
    'Mais de 5 a 10 salários mínimos_homens',
    'Mais de 5 a 10 salários mínimos_mulheres', 'Sem rendimento_homens',
    'Sem rendimento_mulheres'
]

id_col = "Cod"

c_dist_renda = add_lpa_profile_columns(
    df=dist_renda_gen,
    id_col=id_col,
    data_cols=dim_cols,
    prefix="renda",
    divide_by=1000,
    model=fit_lpa_model(
        df=dist_renda_gen,
        data_cols=dim_cols,
        n_profiles=6,
        divide_by=1000,
        covariance_type=covariance_type,
    ),
)

In [59]:
print(f"renda_entropia_classificacao: {c_dist_renda["renda_entropia_classificacao"].mean()}")
print(f"renda_incerteza_normalizada: {c_dist_renda["renda_incerteza_normalizada"].mean()}")
print(f"renda_entropia_bruta: {c_dist_renda["renda_entropia_bruta"].mean()}")
print(f"renda_prob_max: {c_dist_renda["renda_prob_max"].mean()}")

renda_entropia_classificacao: 0.9194574163938029
renda_incerteza_normalizada: 0.08054258360619711
renda_entropia_bruta: 0.14431293685249597
renda_prob_max: 0.9381214818367354


In [60]:
c_dist_renda

,id,Cod,"Brasil, Unidade da Federação e Município",Até 1/4 de salário mínimo_homens,Até 1/4 de salário mínimo_mulheres,Mais de 1 a 2 salários mínimos_homens,Mais de 1 a 2 salários mínimos_mulheres,Mais de 1/2 a 1 salário mínimo_homens,Mais de 1/2 a 1 salário mínimo_mulheres,Mais de 1/4 a 1/2 salário mínimo_homens,...,renda_prob_max,renda_entropia_bruta,renda_incerteza_normalizada,renda_entropia_classificacao,renda_prob_perfil_0,renda_prob_perfil_1,renda_prob_perfil_2,renda_prob_perfil_3,renda_prob_perfil_4,renda_prob_perfil_5
0,0,1,Brasil,33.523582,0.000000,326.437221,0.000000,226.947442,0.000000,53.741396,...,1.000000,2.896570e-07,1.616607e-07,1.000000,1.789627e-13,0.000000,1.524578e-08,0.000000e+00,1.000000e+00,0.000000e+00
1,1,1,Brasil,0.000000,45.102610,0.000000,328.551107,0.000000,260.715225,0.000000,...,0.636189,6.555796e-01,3.658859e-01,0.634114,0.000000e+00,0.363811,0.000000e+00,1.279282e-08,0.000000e+00,6.361892e-01
2,3,11,Rondônia,19.115232,0.000000,350.969218,0.000000,235.981291,0.000000,43.874579,...,0.984740,7.896675e-02,4.407218e-02,0.955928,2.474645e-09,0.000000,1.525963e-02,0.000000e+00,9.847404e-01,0.000000e+00
3,4,11,Rondônia,0.000000,41.745710,0.000000,317.067476,0.000000,274.329471,0.000000,...,0.996716,2.218466e-02,1.238149e-02,0.987619,0.000000e+00,0.003263,0.000000e+00,2.078449e-05,0.000000e+00,9.967160e-01
4,6,12,Acre,57.408119,0.000000,262.170051,0.000000,332.354645,0.000000,97.320266,...,0.998984,8.015957e-03,4.473791e-03,0.995526,1.015756e-03,0.000000,6.574985e-38,0.000000e+00,9.989842e-01,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3471,5206,5221601,Uruaçu (GO),0.000000,40.520680,0.000000,258.135629,0.000000,389.145497,0.000000,...,0.978163,1.073732e-01,5.992612e-02,0.940074,0.000000e+00,0.000470,0.000000e+00,2.136617e-02,0.000000e+00,9.781635e-01
3472,5208,5221858,Valparaíso de Goiás (GO),9.800107,0.000000,406.009745,0.000000,176.790974,0.000000,25.380240,...,0.973911,1.208727e-01,6.746033e-02,0.932540,1.719762e-16,0.000000,9.739109e-01,0.000000e+00,2.608906e-02,0.000000e+00
3473,5209,5221858,Valparaíso de Goiás (GO),0.000000,22.653504,0.000000,399.883599,0.000000,250.061558,0.000000,...,0.715934,5.967533e-01,3.330544e-01,0.666946,0.000000e+00,0.715934,0.000000e+00,2.129916e-08,0.000000e+00,2.840663e-01
3474,5211,5300108,Brasília (DF),12.622670,0.000000,281.254210,0.000000,165.021637,0.000000,28.540788,...,1.000000,-1.000081e-12,-5.581556e-13,1.000000,7.886184e-138,0.000000,1.000000e+00,0.000000e+00,2.984864e-19,0.000000e+00


In [61]:
dist_instucao_gen = pd.read_sql_table("dist_instrucao_genero", con=engine, schema=norm_schema)

dist_instucao_gen = dist_instucao_gen[dist_instucao_gen["populacao"] != "total"].copy()

dist_instucao_gen = pivot_multiplas_colunas_populacao(
    df= dist_instucao_gen,
    categoria_col="populacao",
    populacao_cols=['Sem instrução e fundamental incompleto',
       'Fundamental completo e médio incompleto',
       'Médio completo e superior incompleto', 'Superior completo'],
    rename_categories={
        "homens": "homens",
        "mulheres": "mulheres",
    },
)

In [62]:
dim_cols = [
    'Fundamental completo e médio incompleto_homens',
    'Fundamental completo e médio incompleto_mulheres',
    'Médio completo e superior incompleto_homens',
    'Médio completo e superior incompleto_mulheres',
    'Sem instrução e fundamental incompleto_homens',
    'Sem instrução e fundamental incompleto_mulheres',
    'Superior completo_homens', 'Superior completo_mulheres'
]

In [71]:
id_col = "Cod"

c_dist_instucao = add_lpa_profile_columns(
    df=dist_instucao_gen,
    id_col=id_col,
    data_cols=dim_cols,
    prefix="instrucao",
    divide_by=1000,
    model=fit_lpa_model(
        df=dist_instucao_gen,
        data_cols=dim_cols,
        n_profiles=6,
        divide_by=1000,
        covariance_type=covariance_type,
    ),
)

In [ ]:
print(f"instrucao_entropia_classificacao: {c_dist_instucao["instrucao_entropia_classificacao"].mean()}")
print(f"instrucao_incerteza_normalizada: {c_dist_instucao["instrucao_incerteza_normalizada"].mean()}")
print(f"instrucao_entropia_bruta: {c_dist_instucao["instrucao_entropia_bruta"].mean()}")
print(f"instrucao_prob_max: {c_dist_instucao["instrucao_prob_max"].mean()}")

instrucao_entropia_classificacao: 0.7967212559734553
instrucao_incerteza_normalizada: 0.20327874402654456
instrucao_entropia_bruta: 0.3642266145023471
instrucao_prob_max: 0.8389455002260019


In [72]:
c_dist_instucao

,id,"Brasil, Grande Região, Unidade da Federação e Município",Cod,Fundamental completo e médio incompleto_homens,Fundamental completo e médio incompleto_mulheres,Médio completo e superior incompleto_homens,Médio completo e superior incompleto_mulheres,Sem instrução e fundamental incompleto_homens,Sem instrução e fundamental incompleto_mulheres,Superior completo_homens,...,instrucao_prob_max,instrucao_entropia_bruta,instrucao_incerteza_normalizada,instrucao_entropia_classificacao,instrucao_prob_perfil_0,instrucao_prob_perfil_1,instrucao_prob_perfil_2,instrucao_prob_perfil_3,instrucao_prob_perfil_4,instrucao_prob_perfil_5
0,10000,Campo Alegre,2701407,167.916667,0.0,345.208333,0.0,377.708333,0.0,109.375000,...,0.908453,0.306546,0.171087,0.828913,0.091494,0.0,5.248400e-05,0.0,0.908453,0.0
1,10003,Campos Novos,4203600,152.158427,0.0,362.587263,0.0,354.395213,0.0,130.787862,...,0.779553,0.527474,0.294389,0.705611,0.220447,0.0,5.490722e-08,0.0,0.779553,0.0
2,10005,Canoinhas,4203808,153.823472,0.0,398.663259,0.0,290.102221,0.0,157.361903,...,0.916944,0.286169,0.159714,0.840286,0.916944,0.0,1.009005e-13,0.0,0.083056,0.0
3,10006,Capinzal,4203907,213.094709,0.0,310.137839,0.0,329.368608,0.0,147.398844,...,0.766840,0.543066,0.303091,0.696909,0.233160,0.0,1.642525e-12,0.0,0.766840,0.0
4,10007,Capivari de Baixo,4203956,210.623770,0.0,422.057632,0.0,251.012614,0.0,116.190256,...,0.752832,0.559222,0.312108,0.687892,0.752832,0.0,1.477364e-06,0.0,0.247167,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3741,9987,Bom Jesus,2201903,208.413002,0.0,294.455067,0.0,432.122371,0.0,65.009560,...,0.823602,0.521288,0.290936,0.709064,0.016766,0.0,1.596321e-01,0.0,0.823602,0.0
3742,9991,Braço do Norte,4202800,218.706157,0.0,374.746687,0.0,296.258769,0.0,110.288387,...,0.587543,0.677836,0.378307,0.621693,0.412449,0.0,8.074658e-06,0.0,0.587543,0.0
3743,9994,Brusque,4202909,209.989787,0.0,385.350078,0.0,247.758823,0.0,156.901313,...,0.578345,0.680820,0.379973,0.620027,0.578345,0.0,8.924664e-14,0.0,0.421655,0.0
3744,9995,Caçador,4203006,187.592047,0.0,301.399116,0.0,365.243004,0.0,145.729013,...,0.819136,0.472702,0.263820,0.736180,0.180864,0.0,8.329832e-12,0.0,0.819136,0.0


In [ ]:
from sqlalchemy import text

def drop_schema(
    engine: Engine,
    schema_name: str,
) -> None:
    preparer = engine.dialect.identifier_preparer
    quoted_schema = preparer.quote_schema(schema_name)

    with engine.begin() as connection:
        connection.execute(text(f"DROP SCHEMA IF EXISTS {quoted_schema} CASCADE"))

    inspector = inspect(engine)
    inspector.clear_cache()
    if inspector.has_schema(schema_name):
        raise RuntimeError(f"Schema still exists after DROP SCHEMA CASCADE: {schema_name}")

    print(f"Dropped schema and all objects inside it: {schema_name}")


# Para usar quando quiser recomeçar:
drop_schema(engine, target_schema_name)


with engine.begin() as conn:
    conn.execute(text(f'CREATE SCHEMA IF NOT EXISTS "{"clusters"}";'))

In [77]:
c_dist_idade.to_sql(
    name= "dist_idade",
    con=engine,
    schema="clusters",
    if_exists="replace"
)

555

In [78]:
c_dist_renda.to_sql(
    name= "dist_renda",
    con=engine,
    schema="clusters",
    if_exists="replace"
)

827

In [79]:
c_dist_instucao.to_sql(
    name= "dist_instrucao",
    con=engine,
    schema="clusters",
    if_exists="replace"
)

746